In [0]:
dbutils.widgets.text("catalog","default")
catalog=dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df = spark.read.table(f"{catalog}.bronze.sessions")

# Helper: empty string -> NULL
def nullif_empty(c):
    return when(c == '', lit(None).cast('string')).otherwise(c)

# S1 - session_id, charger_id, member_id: trim, upper; empty -> NULL
df = df.withColumn('session_id', nullif_empty(trim(upper(col('session_id'))))) \
       .withColumn('charger_id', nullif_empty(trim(upper(col('charger_id'))))) \
       .withColumn('member_id', nullif_empty(trim(upper(col('member_id')))))

# S2 - status: trim, upper case
df = df.withColumn('status', trim(upper(col('status'))))

# S3 - started_at, ended_at: valid as yyyy-MM-dd HH:mm:ss or yyyy-MM-ddTHH:mm:ss; else NULL
#       session_date = date of started_at
started_at_ts = coalesce(
    try_to_timestamp(col('started_at'), lit('yyyy-MM-dd HH:mm:ss')),
    try_to_timestamp(col('started_at'), lit("yyyy-MM-dd'T'HH:mm:ss"))
)
ended_at_ts = coalesce(
    try_to_timestamp(col('ended_at'), lit('yyyy-MM-dd HH:mm:ss')),
    try_to_timestamp(col('ended_at'), lit("yyyy-MM-dd'T'HH:mm:ss"))
)

df = df.withColumn('started_at', started_at_ts) \
       .withColumn('ended_at', ended_at_ts) \
       .withColumn('session_date', to_date(col('started_at')))

# S4 - energy_kwh = meter.end_kwh - meter.start_kwh when both exist, else NULL
df = df.withColumn(
    'energy_kwh',
    when(
        col('meter.end_kwh').isNotNull() & col('meter.start_kwh').isNotNull(),
        col('meter.end_kwh') - col('meter.start_kwh')
    ).otherwise(lit(None).cast('double'))
)

# S5 - duration_minutes = whole minutes from started_at to ended_at, rounded down
#       NULL if ended_at is missing or invalid
df = df.withColumn(
    'duration_minutes',
    when(
        col('started_at').isNotNull() & col('ended_at').isNotNull(),
        floor((col('ended_at').cast('long') - col('started_at').cast('long')) / 60)
    ).otherwise(lit(None).cast('int'))
)

# S6 - time_band = PEAK if hour of started_at is 18-21, else OFF_PEAK
df = df.withColumn(
    'time_band',
    when(col('started_at').isNotNull() & hour(col('started_at')).between(18, 21), lit('PEAK'))
    .when(col('started_at').isNotNull(), lit('OFF_PEAK'))
    .otherwise(lit(None).cast('string'))
)

# S7 - idle_minutes: default to 0 when missing/NULL
# S8 - fault_code (in faults array): trim, upper case
df = df.withColumn('idle_minutes', coalesce(col('idle_minutes'), lit(0))) \
       .withColumn(
           'faults',
           transform(col('faults'), lambda x: x.withField('code', trim(upper(x['code']))))
       )

In [0]:
# Keep one version per session_id: the latest updated_at

from pyspark.sql.window import Window
w = Window.partitionBy("session_id").orderBy(col("updated_at").desc())
df = df.withColumn("rank", row_number().over(w)).filter(col("rank") == 1).drop("rank")

In [0]:
# C3 – Validation: route each session to silver.sessions or silver.quarantine_sessions
# Only the latest version of each session (output of C2) is validated.
# A session is rejected if it breaks ANY rule; the rejection reason is the
# lowest-numbered broken rule.  A rejected session goes to quarantine as a
# whole — none of its faults reach Silver.  A session is in silver.sessions
# OR silver.quarantine_sessions, never both.

from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ── Reference data for lookup validation (R1, R2) ───────────────────
chargers_ref = (
    spark.read.table(f"{catalog}.silver.chargers")
    .select(col("charger_id").alias("_ref_charger"))
    .distinct()
)

members_ref = (
    spark.read.table(f"{catalog}.silver.members")
    .filter(col("is_current") == True)
    .select(col("member_id").alias("_ref_member"))
    .distinct()
)

# ── Session-level reference joins (left join → null = unknown) ──────
df_v = (df
    .join(broadcast(chargers_ref), df["charger_id"] == col("_ref_charger"), "left")
    .join(broadcast(members_ref), df["member_id"] == col("_ref_member"), "left"))

# ── Rules R1-R5 ────────────────────────────────────────────────────
df_v = (df_v
    .withColumn("r1", when(col("charger_id").isNull() | col("_ref_charger").isNull(), 1).otherwise(0))
    .withColumn("r2", when((col("r1") == 0) & col("member_id").isNotNull() & col("_ref_member").isNull(), 1).otherwise(0))
    .withColumn("r3", when(col("started_at").isNull(), 1).otherwise(0))
    .withColumn("r4", when(
        col("status").isin("COMPLETED", "FAILED") &
        (col("ended_at").isNull() | (col("ended_at") <= col("started_at"))), 1).otherwise(0))
    .withColumn("r5", when(
        (col("status") == "COMPLETED") &
        (col("energy_kwh").isNull() | (col("energy_kwh") < 0)), 1).otherwise(0)))

# ── Determine rejection reason (lowest-numbered broken rule) ───────
df_v = df_v.withColumn("reject_reason",
    when(col("r1") == 1, "UNKNOWN_CHARGER")
    .when(col("r2") == 1, "UNKNOWN_MEMBER")
    .when(col("r3") == 1, "INVALID_TIMESTAMP")
    .when(col("r4") == 1, "INVALID_DURATION")
    .when(col("r5") == 1, "INVALID_METER")
    .otherwise(lit(None)))

# ── Split: valid → silver, rejected → quarantine ───────────────────
_helper_cols = ["r1", "r2", "r3", "r4", "r5", "_ref_charger", "_ref_member"]

valid_df  = df_v.filter(col("reject_reason").isNull()).drop(*_helper_cols)
reject_df = df_v.filter(col("reject_reason").isNotNull()).drop(*_helper_cols)

# ── Prepare raw_record (full Bronze row as JSON) for quarantine ─────
bronze_raw = spark.read.table(f"{catalog}.bronze.sessions")
bronze_json = (
    bronze_raw
    .withColumn("raw_record", to_json(struct("*")))
    .withColumn("_sid", trim(upper(col("session_id"))))
    .withColumn("_rn", row_number().over(
        Window.partitionBy("_sid").orderBy(col("updated_at").desc())))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .select(col("_sid").alias("session_id"), "raw_record"))

# ── C4: Output tables ──────────────────────────────────────────────

# Silver sessions (valid sessions)
silver_sessions = (
    valid_df.select(
        col("session_id"),
        col("charger_id"),
        col("member_id"),
        col("started_at"),
        col("ended_at"),
        col("session_date"),
        col("status"),
        col("time_band"),
        col("energy_kwh").cast("decimal(10,3)").alias("energy_kwh"),
        col("duration_minutes").cast("int").alias("duration_minutes"),
        col("idle_minutes").cast("int").alias("idle_minutes"),
        col("payment.amount_inr").cast("decimal(10,2)").alias("amount_paid"),
        try_to_timestamp(col("updated_at")).alias("updated_at")))

# Silver session_faults (one row per fault of a valid session)
silver_session_faults = (
    valid_df
    .select("session_id", posexplode(col("faults")))
    .withColumn("fault_seq", col("pos") + 1)
    .withColumn("fault_code", col("col.code"))
    .select("session_id", "fault_seq", "fault_code"))

# Quarantine sessions (rejected sessions)
quarantine_sessions = (
    reject_df
    .join(bronze_json, "session_id", "left")
    .select(
        col("session_id"),
        col("reject_reason"),
        current_timestamp().alias("quarantined_at"),
        col("raw_record")))

# ── C5: Incremental maintenance with MERGE INTO (idempotent) ───────
def merge_or_create(table_full, source_df, merge_on):
    """Create on first run; MERGE INTO on subsequent runs.
    WHEN NOT MATCHED BY SOURCE DELETE removes sessions that moved between
    valid and rejected states, keeping the two tables mutually exclusive."""
    if not spark.catalog.tableExists(table_full):
        source_df.write.format("delta").saveAsTable(table_full)
    else:
        source_df.createOrReplaceTempView("merge_src")
        spark.sql(f"""
            MERGE INTO {table_full} AS t
            USING merge_src AS s
            ON {merge_on}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
            WHEN NOT MATCHED BY SOURCE THEN DELETE
        """)

merge_or_create(f"{catalog}.silver.sessions", silver_sessions,
                "t.session_id = s.session_id")
merge_or_create(f"{catalog}.silver.session_faults", silver_session_faults,
                "t.session_id = s.session_id AND t.fault_seq = s.fault_seq")
merge_or_create(f"{catalog}.silver.quarantine_sessions", quarantine_sessions,
                "t.session_id = s.session_id")

print(f"silver.sessions:           {spark.read.table(f'{catalog}.silver.sessions').count()} sessions")
print(f"silver.session_faults:     {spark.read.table(f'{catalog}.silver.session_faults').count()} faults")
print(f"silver.quarantine_sessions: {spark.read.table(f'{catalog}.silver.quarantine_sessions').count()} sessions")

display(spark.sql(f"SELECT * FROM {catalog}.silver.sessions LIMIT 10"))